# CAMeL-BERT Inference & Evaluation

Run token classification on full corpus and evaluate boundary precision.

**Expected**: 65-75% usable boundaries (IoU >= 0.80) vs baseline 3.1% = **20-25x improvement**

## STAGE 1: Load Model

In [ ]:
print("[STAGE 1] Loading trained model...\n")

from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import json
from pathlib import Path
import numpy as np
from collections import defaultdict

# Load model
model_path = Path('/content/drive/MyDrive/Khabar-segmentation/data/camelbert_training/best_model')
tokenizer = AutoTokenizer.from_pretrained('aubmindlab/bert-base-arabertv2')
model = AutoModelForTokenClassification.from_pretrained(str(model_path))

# Load label mapping
with open(model_path / 'label_mapping.json', 'r', encoding='utf-8') as f:
    label_data = json.load(f)
    id2label = {int(k): v for k, v in label_data['id2label'].items()}

print(f"[OK] Model loaded with {len(model.config.id2label)} labels")
print(f"Labels: {id2label}\n")

## STAGE 2: Load Corpus & Reference Boundaries

In [ ]:
print("[STAGE 2] Loading reference corpus...")

corpus_path = Path('/content/drive/MyDrive/Khabar-segmentation/data/processed/kitab_uqala_reference_corpus.txt')
with open(corpus_path, 'r', encoding='utf-8') as f:
    full_corpus = f.read()

print(f"[OK] Corpus loaded: {len(full_corpus)} characters\n")

print("[STAGE 3] Loading reference boundaries...")

boundaries_path = Path('/content/drive/MyDrive/Khabar-segmentation/data/processed/kitab_uqala_boundaries.json')
with open(boundaries_path, 'r', encoding='utf-8') as f:
    boundaries_data = json.load(f)

# Handle both list and dict structures
if isinstance(boundaries_data, dict):
    ref_boundaries = boundaries_data.get('boundaries', boundaries_data.get('akhbars', []))
elif isinstance(boundaries_data, list):
    ref_boundaries = boundaries_data
else:
    ref_boundaries = []

print(f"[OK] Loaded {len(ref_boundaries)} reference boundaries")
print(f"Sample: {ref_boundaries[0] if ref_boundaries else 'empty'}\n")

## STAGE 4: Run Inference on Corpus

In [ ]:
print("[STAGE 4] Running inference on corpus...\n")

# Create pipeline
nlp = pipeline('token-classification', 
               model=model, 
               tokenizer=tokenizer,
               aggregation_strategy='simple',
               device=0)

# Process corpus in chunks with stride
chunk_size = 512
chunk_stride = 400
all_predictions = []

char_pos = 0
chunk_num = 0

while char_pos < len(full_corpus):
    chunk_end = min(char_pos + chunk_size, len(full_corpus))
    chunk_text = full_corpus[char_pos:chunk_end]
    
    try:
        predictions = nlp(chunk_text)
        
        # Store predictions with absolute character positions
        for pred in predictions:
            all_predictions.append({
                'entity': pred['entity_group'],
                'score': float(pred['score']),
                'word': pred['word'],
                'char_start': char_pos + pred['start'],
                'char_end': char_pos + pred['end']
            })
        
        chunk_num += 1
        if chunk_num % 20 == 0:
            print(f"  Processed {chunk_num} chunks ({chunk_end}/{len(full_corpus)} chars)")
            
    except Exception as e:
        print(f"  [ERROR] Chunk at {char_pos}: {e}")
    
    char_pos = chunk_end - chunk_stride
    if chunk_end >= len(full_corpus):
        break

print(f"\n[OK] Inference complete: {len(all_predictions)} tokens classified\n")

## STAGE 5: Extract & Match Boundaries

In [ ]:
print("[STAGE 5] Extracting predicted boundaries...\n")

def compute_iou(pred_start, pred_end, ref_start, ref_end):
    """Compute Intersection over Union between two character ranges."""
    intersection = max(0, min(pred_end, ref_end) - max(pred_start, ref_start))
    union = max(pred_end, ref_end) - min(pred_start, ref_start)
    if union == 0:
        return 0.0
    return intersection / union

# Extract boundaries where new segment starts
pred_boundaries = []
for pred in all_predictions:
    if pred['entity'] in ['B-ISNAD', 'B-KHABAR']:
        pred_boundaries.append({
            'char_pos': pred['char_start'],
            'type': 'ISNAD' if pred['entity'] == 'B-ISNAD' else 'KHABAR',
            'score': pred['score']
        })

print(f"Found {len(pred_boundaries)} predicted boundaries")
print(f"  - ISNAD starts: {sum(1 for b in pred_boundaries if b['type'] == 'ISNAD')}")
print(f"  - KHABAR starts: {sum(1 for b in pred_boundaries if b['type'] == 'KHABAR')}\n")

# Match with reference boundaries
print("Matching predicted vs reference boundaries...\n")

matches = []
matched_refs = set()

for pred in pred_boundaries:
    best_iou = 0
    best_ref_idx = -1
    
    for ref_idx, ref in enumerate(ref_boundaries):
        # Get reference position
        ref_pos = ref.get('char_pos') or ref.get('isnad_end')
        if ref_pos is None:
            continue
        
        # Compute IoU around boundary position (within 50 char window)
        iou = compute_iou(pred['char_pos']-25, pred['char_pos']+25,
                         ref_pos-25, ref_pos+25)
        
        if iou > best_iou:
            best_iou = iou
            best_ref_idx = ref_idx
    
    if best_ref_idx >= 0:
        matches.append({
            'pred_pos': pred['char_pos'],
            'ref_pos': ref_boundaries[best_ref_idx].get('char_pos') or ref_boundaries[best_ref_idx].get('isnad_end'),
            'iou': best_iou,
            'score': pred['score']
        })
        matched_refs.add(best_ref_idx)

print(f"Matched {len(matches)}/{len(ref_boundaries)} reference boundaries\n")

## STAGE 6: Compute Evaluation Metrics

In [ ]:
print("[STAGE 6] Computing metrics...\n")

# Extract IoU scores
ious = [m['iou'] for m in matches]
mean_iou = np.mean(ious) if ious else 0.0
median_iou = np.median(ious) if ious else 0.0

# Count usable boundaries (IoU >= 0.80)
usable = sum(1 for m in matches if m['iou'] >= 0.80)
pct_usable = 100 * usable / len(ref_boundaries) if ref_boundaries else 0

# Boundary errors
errors = [abs(m['pred_pos'] - m['ref_pos']) for m in matches]
mean_error = np.mean(errors) if errors else 0
median_error = np.median(errors) if errors else 0

# Compare with baseline
baseline_usable = 3.1
improvement = pct_usable / baseline_usable if baseline_usable > 0 else 0

print("="*80)
print("EVALUATION RESULTS")
print("="*80)
print(f"\nBOUNDARY DETECTION")
print(f"  Detected: {len(matches)}/{len(ref_boundaries)} ({100*len(matches)/len(ref_boundaries):.1f}%)")
print(f"  Usable (IoU >= 0.80): {usable}/{len(ref_boundaries)} ({pct_usable:.1f}%)")
print(f"\nBOUNDARY PRECISION")
print(f"  Mean IoU: {mean_iou:.3f}")
print(f"  Median IoU: {median_iou:.3f}")
print(f"\nBOUNDARY POSITION ACCURACY")
print(f"  Mean error: {mean_error:.1f} characters")
print(f"  Median error: {median_error:.1f} characters")
print(f"\nCOMPARISON WITH BASELINE")
print(f"  Baseline v3.5: {baseline_usable:.1f}% usable")
print(f"  CAMeL-BERT: {pct_usable:.1f}% usable")
print(f"  Improvement: {improvement:.1f}x\n")
print("="*80)

# IoU distribution
print(f"\nIOU DISTRIBUTION")
for threshold in [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]:
    count = sum(1 for m in matches if m['iou'] >= threshold)
    pct = 100*count/len(matches) if matches else 0
    print(f"  IoU >= {threshold}: {count}/{len(matches)} ({pct:.1f}%)")

print(f"\n[OK] Evaluation complete")

## STAGE 7: Save Results

In [ ]:
print("[STAGE 7] Saving evaluation results...\n")

output_dir = Path('/content/drive/MyDrive/Khabar-segmentation/data/camelbert_training/inference_results')
output_dir.mkdir(parents=True, exist_ok=True)

# Save detailed metrics
metrics = {
    'total_boundaries': len(ref_boundaries),
    'detected': len(matches),
    'detection_rate': 100 * len(matches) / len(ref_boundaries),
    'usable_count': usable,
    'usable_percentage': pct_usable,
    'mean_iou': float(mean_iou),
    'median_iou': float(median_iou),
    'mean_error_chars': float(mean_error),
    'median_error_chars': float(median_error),
    'baseline_usable': baseline_usable,
    'improvement_factor': float(improvement),
    'iou_distribution': {
        '>=0.50': sum(1 for m in matches if m['iou'] >= 0.50),
        '>=0.60': sum(1 for m in matches if m['iou'] >= 0.60),
        '>=0.70': sum(1 for m in matches if m['iou'] >= 0.70),
        '>=0.80': sum(1 for m in matches if m['iou'] >= 0.80),
        '>=0.90': sum(1 for m in matches if m['iou'] >= 0.90),
        '>=0.95': sum(1 for m in matches if m['iou'] >= 0.95),
    }
}

with open(output_dir / 'detailed_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print(f"[OK] Saved metrics to: {output_dir}/detailed_metrics.json")

print(f"\n[DONE] All results saved to Drive")